# 📗 Práctica Integradora — Sesión 3
## Versión Estudiante · Dashboard HR con diseño visual profesional
---


## 🎯 Objetivo de aprendizaje

Al finalizar esta práctica serás capaz de:

- Aplicar CSS personalizado para estilizar la app con la paleta corporativa de TechNova
- Estructurar la navegación con `ui.page_navbar()` y múltiples pestañas
- Implementar 4 tarjetas KPI con semáforo de colores condicionales calculados en Python
- Agregar una alerta dinámica que aparece cuando los datos son insuficientes para análisis

**Nivel Bloom:** Aplicar · Crear  
**Punto de partida:** `app.py` de la Sesión 2 con `@reactive.calc` y 2 gráficas plotly  
**Producto esperado:** `app.py` con CSS corporativo, `navbarPage`, 4 KPIs con semáforo y alerta condicional


## 🏢 Contexto — TechNova Solutions S.A. de C.V.

| Dato | Valor |
|------|-------|
| Sector | Tecnología de la información |
| Colaboradores | 1,470 en 3 departamentos |
| Tasa de rotación actual | **16.1%** (sector: 11%) |
| Costo por baja | **$2,800 USD** por colaborador |
| Departamento crítico | HR con **25.9%** de rotación |

**Tu rol:** Eres el **Analista de Datos** del área de RH de TechNova Solutions.  
Tu cliente interno es el **Lic. Marcos Andrade**, Director de RH.


## 📋 Escenario — Semana 3: Presentación a la Dirección General

El Lic. Andrade convoca una reunión urgente:

> *"La Dirección General quiere ver el dashboard la próxima semana.  
> Necesito que se vea profesional — los colores de la empresa,  
> pestañas para organizarlo mejor, y que los números críticos salten a la vista.  
> Si la rotación es muy alta que se vea en rojo.  
> Y pon un aviso cuando el filtro sea tan específico  
> que los porcentajes ya no sean confiables."*

**Lo que debes entregar al final de esta sesión:**  
Transformar el `app.py` de la Semana 2 en un dashboard de presentación con:

1. **CSS corporativo** — navbar azul oscuro `#1F3864`, tarjetas KPI con sombra, fuente Segoe UI
2. **`ui.page_navbar()`** — con 2 pestañas: "Resumen" y "Detalle empleados"
3. **4 tarjetas KPI** con semáforo de colores condicionales calculados en Python
4. **Alerta dinámica** — caja naranja cuando los filtros dejan menos de 10 colaboradores

> 💡 **Regla del semáforo:**  
> El color se calcula en **Python** e inyecta en `style=` del HTML.  
> No puede hacerse en CSS puro porque depende del valor calculado en el servidor.  
> **Umbrales de rotación:** Verde ≤10% · Naranja 10-20% · Rojo >20%

---
> El `app.py` que construyas hoy es la plantilla base del dashboard final de la Sesión 4.


## 📗 Instrucciones para el estudiante

### ¿Qué vas a construir?

El mismo dashboard de la Semana 2 transformado en una presentación profesional:

| Elemento nuevo | Descripción |
|----------------|-------------|
| **CSS corporativo** | Navbar azul `#1F3864`, tarjetas KPI con sombra, fuente Segoe UI |
| **`navbarPage`** | 2 pestañas: "📊 Resumen" y "👥 Detalle empleados" |
| **4 KPIs** | Total colaboradores, Rotación (semáforo), Salario promedio, Departamento crítico |
| **Semáforo** | 🟢 Verde ≤10% · 🟠 Naranja 10-20% · 🔴 Rojo >20% |
| **Alerta** | Caja naranja si el filtro deja menos de 10 colaboradores |

### Regla del semáforo

El color se calcula en **Python** e inyecta en `style=` del HTML:

```python
color = '#C62828' if pct > 20 else ('#F57F17' if pct > 10 else '#2E7D32')
```

No puede hacerse en CSS puro — CSS no conoce el valor calculado en el servidor.


---
## Celda 1 — CSS corporativo de TechNova

In [ ]:
# ▶ EJECUTA — Celda 1: CSS corporativo de TechNova
# ──────────────────────────────────────────────────────────────
# El CSS se almacena como lista de strings y se une con join()
# para evitar problemas de comillas anidadas.

import pandas as pd
import plotly.express as px
from shiny import App, ui, render, reactive

df = pd.read_csv('../../hr_ibm.csv')

#aqui van los controles
opts_dept = ["Todos"] + sorted(df['Department'].unique().tolist())
opts_attr = ["Todos", "Yes", "No"]


#definimos la css
css_lines = [
    ' body { background-color: #F4F6F9; font-family: Segou UI, Arial, sans-serif;  }  ',
    ' .navbar { background-color: #1F3864!important;} ',
    ' .navbar-brand, .nav-link { color: white!important; font-weight: 500; } ',
    ' .sidebar { backgroung: white; border-rigth: 3px solid #2E5FA3; border-radius: 8px; padding: 16px; } ',
    ' .kpi-card { background: white; border-radius:10px; padding:20px; text-align: center; box-shadow: 0 2px 8px rgba(0,0,0,0.1); } ',
    ' .kpi-valor { font-size:2em, font-weight:bold; } ',
    ' .kpi-label { font-size: .85em; color: #666; margin-top: 4px; } ',
    ' .h3 { color: #1F3864; border-bottom: 2px solid "#2EFA35; padding-bottom: 6px} '
]

css = ui.tags.style('\n'.join(css_lines))


## Celda 2 — `navbarPage` con pestañas

In [ ]:
# ▶ EJECUTA — Celda 2: navbarPage con pestañas de navegación
# ──────────────────────────────────────────────────────────────
# ui.page_navbar() reemplaza ui.page_fluid() de la Sesión 2.
# Los filtros del sidebar siguen afectando todas las pestañas
# porque el @reactive.calc del server es compartido.

app_ui_c2 = ui.page_navbar(
    css,
    ui.nav_panel(
        'Resumen',
        ui.layout_sidebar(
            ui.sidebar(
                ui.h5('Filtros'),
                ui.input_select('dept', 'Department', choices=opts_dept),
                ui.input_select('attrition', 'Rotación', choices=opts_attr),
                ui.input_slider('edad', 'Rango de Edad', min=18, max=60, value=[18,60]),
                ui.input_action_button('btn_reset', 'Restablecer', class_='btn-sm mt-2'),
            ),
            ui.output_ui('alerta_datos'),
            ui.h3('Indicadores clave'),
            ui.row(
                ui.column(3, ui.output_ui('kpi_total')),
                ui.column(3, ui.output_ui('kpi_rotacion')),
                ui.column(3, ui.output_ui('kpi_salario')),
                ui.column(3, ui.output_ui('kpi_depto')),
            ),
            ui.h3('Graficas o Visualizaciones'),
            ui.row(
                ui.column(6, ui.output_ui('grafica_rotacion')),
                ui.column(6, ui.output_ui('grafica_scatter')),
            ),
        )
    ),
    ui.nav_panel('Detalle de Empleados', ui.output_table('tabla_detalle')),
    title='Dashboard General de TechNova Solution Departamento de RH'
)




## Celda 3 — KPI de rotación con semáforo ✏️

In [ ]:
# ✏️ TU TURNO — Celda 3: Crea el KPI de rotación con semáforo
# ──────────────────────────────────────────────────────────────
# El color depende del valor calculado en Python — no en CSS puro.
# Semáforo TechNova:
#   Verde:   pct <= 10%  → situación aceptable
#   Naranja: pct 10-20%  → zona de alerta
#   Rojo:    pct > 20%   → requiere acción inmediata del Director

# Fragmento del server — va dentro de def server(input, output, session):

    # En ui: ui.output_ui('kpi_rotacion') dentro de ui.column(3, ...)

def _kpis_ejemplo(datos_filtrados):

    def kpi_total():
        n = len(datos_filtrados())
        return ui.div(
            ui.div(f'{n:,}', class_='kpi-valor', style='color:#1F3864'),
            ui.div('Total de colaboradores: ', class_='kpi-label'), class_='kpi-card'
        )

## Celda 4 — Histograma de distribución salarial

In [ ]:
# ▶ EJECUTA — Celda 4: Histograma de distribución salarial
# ──────────────────────────────────────────────────────────────
# Muestra si los que rotan se concentran en tramos salariales bajos.
def grafica_salario_demo(datos_filtrados):

    def grafica_salario():
        d = datos_filtrados()
        fig = px.histogram(
            d, 
            x='MonthlyIncome',
            color='Attrition',
            color_discrete_map={'Yes':'#E59395', 'No': '#431047'},
            nbins=30,
            barmode='overlay', #se superponen
            opacity=0.75,
            title='Distribución de Salario Mensual por estado de rotación',
            labels={'MonthlyIncome': 'Ingreso Mensual (USD)', 'Attrition' : 'Rotación'}
        )
        fig.update_layout(height=350)
        return ui.HTML(fig.to_html(include_plotlyjs='cdn'))


## Celda 5 — Alerta de datos insuficientes

In [ ]:
# ▶ EJECUTA — Celda 5: Alerta de datos insuficientes
# ──────────────────────────────────────────────────────────────
# Cuando el filtro deja menos de 10 colaboradores, aparece
# una caja de advertencia. Con 10 o más, desaparece.

def alerta_demo(datos_filtrados):

    def alerta_datos():
        n = len(datos_filtrados())
        if n < 10:
            return ui.div(
                ui.strong('Advertencia: '), 
                f'Solo {n} colaboradores con los filtros actuales.'
                ' Los porcentajes pueden no ser estadisticamente reprensentativos',
                style=(
                    'background:#FFF3E0; border-left:4px solid #E65100;'
                    'padding:10px 14px; boder-radius:4px;'
                    'margin:8px 0; color:#E65100'
                )
            )
        return ui.div()



## Celda 6 — App completa: dashboard de presentación ✏️

In [ ]:
# ✏️ TU TURNO — Celda 6: App completa — Dashboard de presentación
# ──────────────────────────────────────────────────────────────
# Integra todo lo de las celdas anteriores.
# Esta versión la presentarás a la Dirección General.
#
# Checklist antes de ejecutar:
#   [ ] CSS corporativo como primer argumento en page_navbar
#   [ ] 2 pestañas: Resumen y Detalle empleados
#   [ ] 4 KPIs con semáforo en kpi_rotacion
#   [ ] Alerta de datos insuficientes
#   [ ] 3 gráficas plotly reactivas
#   [ ] shiny run app.py sin errores


# Guardar como app.py → shiny run app.py

## ✅ Autoevaluación — Práctica Integradora Sesión 3

Verifica estos puntos **antes de entregar** tu `app.py`:

| N° | Criterio | ✅ Cumple | ❌ Pendiente |
|----|----------|----------|------------|
| 1 | El CSS aplica correctamente: navbar azul `#1F3864`, fondo gris, kpi-card con sombra | | |
| 2 | La app usa `ui.page_navbar()` con al menos 2 pestañas funcionales | | |
| 3 | Los 4 KPIs aparecen en una fila de 4 columnas | | |
| 4 | El KPI de rotación cambia de color según el semáforo (verde/naranja/rojo) | | |
| 5 | El color del KPI se actualiza al cambiar los filtros | | |
| 6 | La alerta de datos insuficientes aparece con menos de 10 colaboradores | | |
| 7 | `shiny run app.py` corre sin errores en la terminal | | |
| 8 | La app tiene al menos 3 gráficas plotly interactivas | | |

---

### Preguntas de análisis — Responde usando tu app

**1.** Filtra solo el departamento HR. El KPI de rotación, ¿en qué color aparece?  
¿Qué significado tiene ese color para el Director de RH?

> _________________________________________________________________

**2.** Con todos los filtros en "Todos", ¿cuál es el departamento crítico según el KPI?

> _________________________________________________________________

**3.** Filtra edades entre 50 y 60 años. ¿Aparece la alerta?  
¿Cuántos colaboradores hay en ese rango?

> _________________________________________________________________
